# stanford to csv
loads the hein-daily congressional speech files and produces a labeled csv
covering congresses 104-114 (1995-2016)

In [ ]:
import os
import csv
import pandas as pd
# update paths
BASE_DIR=    '/kaggle/input/datasets/zainabrizvi5/thesis-data/hein-daily'
OUTPUT_FILE= 'stanford_labeled.csv'

In [ ]:
# D = democrat = left, D = republican = right
PARTY_TO_LABEL= {
    'D': 'left',
    'R': 'right',
}
CONGRESSES= range(104, 115)

In [ ]:
def load_speakermap(filepath):
    # maps speech_id to party from the speakermap file
    speech_to_party= {}
    with open(filepath, 'r', encoding ='utf-8', errors='replace') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts= line.split('|')
            if len(parts) < 8: continue
            speech_id= parts[1].strip()
            party = parts[7].strip()
            if party in PARTY_TO_LABEL:
                speech_to_party[speech_id] = party
    return speech_to_party

def load_speeches(filepath):
    # maps speech_id to text
    speech_to_text = {}
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        next(f)
        for line in f:
            line= line.strip()
            if not line: continue
            idx= line.find('|')
            if idx ==-1: continue
            speech_id= line[:idx].strip()
            text= line[idx+1:].strip()
            if text:
                speech_to_text[speech_id]= text
    return speech_to_text

In [ ]:
def process_congress(congress_num, base_dir):
    folder= os.path.join(base_dir, str(congress_num).zfill(3))
    if not os.path.isdir(folder):
        print(f'  skipping {folder} - not found')
        return []

    speakermap_file= None
    speeches_file= None
    for fname in os.listdir(folder):
        if 'speakermap' in fname.lower() or 'speaker_map' in fname.lower():
            speakermap_file= os.path.join(folder, fname)
        elif fname.startswith('speeches_') and fname.endswith('.txt'):
            speeches_file= os.path.join(folder, fname)

    if not speakermap_file or not speeches_file:
        print(f'  skipping {folder} : missing files')
        return []

    speech_to_party= load_speakermap(speakermap_file)
    speech_to_text= load_speeches(speeches_file)

    rows= []
    
    for speech_id, text in speech_to_text.items():
        party= speech_to_party.get(speech_id)
        if not party: continue
        rows.append({'speech_id': speech_id,'congress': congress_num,'text': text,'party': party,'label': PARTY_TO_LABEL[party],'word_count': len(text.split()),})
    return rows

In [ ]:
all_rows= []
print(f'loading from {BASE_DIR}')
for congress in CONGRESSES:
    print(f'congress {congress}')
    rows= process_congress(congress, BASE_DIR)
    all_rows.extend(rows)
    print(f'  {len(rows):,} speeches')
print(f'\ntotal: {len(all_rows):,}')

In [ ]:
# write to csv
fieldnames= ['speech_id','congress','text','party','label','word_count']
with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
    writer= csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_rows)
df= pd.DataFrame(all_rows)
print(df['label'].value_counts())
print(f'saved to {OUTPUT_FILE}')